# TPU Benchmark Harness on Colab Pro

Runs the same smoke/quick suites as `./scripts/provision_tpu.sh` flow, but on Colab Pro's TPU at $0 marginal cost.
Use for development iteration; reserve GCP v5e-1 for measurement-grade runs.
See `DECISIONS.md` ADR-006 + todo.md Tier 3 #9.

Before running: **Runtime → Change runtime type → TPU**.
Output is identical to the gcloud path (same `results/runs.jsonl`, same `results/otel/*.jsonl`).

In [ ]:
import os, subprocess, sys
# Detect Colab + TPU
try:
    import google.colab     # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Clone (idempotent)
if not os.path.exists('/content/tpu'):
    !git clone --depth=1 https://github.com/rajaghv-dev/tpu.git /content/tpu
%cd /content/tpu

# JAX TPU check — if this fails the runtime isn't TPU
import jax
print(f"jax {jax.__version__} | devices: {jax.devices()}")
assert any('TPU' in str(d) for d in jax.devices()), "Select Runtime → Change runtime type → TPU"

In [ ]:
!pip install -q -r requirements.txt
# transformers / flax / opentelemetry usually need installing
!pip install -q 'transformers>=4.40,<4.45' 'flax>=0.8.3' 'opentelemetry-sdk>=1.27' 'opentelemetry-exporter-otlp-proto-grpc>=1.27'

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/tpu'
os.environ['TPU_BENCH_OTEL'] = 'file'   # write OTLP-JSON locally; user downloads + replays on laptop
os.environ['TPU_BENCH_OTEL_DIR'] = '/content/tpu/results/otel'
os.environ['JAX_COMPILATION_CACHE_DIR'] = '/content/xla-cache'  # ephemeral per Colab session but speeds re-runs
os.makedirs('/content/tpu/results/otel', exist_ok=True)
os.makedirs('/content/xla-cache', exist_ok=True)
print('OTel dir   :', os.environ['TPU_BENCH_OTEL_DIR'])
print('XLA cache  :', os.environ['JAX_COMPILATION_CACHE_DIR'])

## HuggingFace auth (gated models: Gemma / LLaMA / PaliGemma)

Open the **Secrets** panel (key icon, left sidebar) and add:
- Name: `HF_TOKEN`
- Value: your HF PRO token from https://huggingface.co/settings/tokens (read scope)
- Enable **Notebook access**

The cell below reads it via `google.colab.userdata` and installs it where `transformers` / `huggingface_hub` will find it. If `HF_TOKEN` is missing the cell warns but does not fail — non-gated models (BERT, ViT, GPT-2, Whisper, CLIP) work without it.

In [ ]:
import os, pathlib
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    # Canonical HF location — transformers + huggingface_hub auto-read.
    for p in [pathlib.Path.home() / '.cache' / 'huggingface' / 'token',
              pathlib.Path.home() / '.huggingface' / 'token']:
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(HF_TOKEN)
        p.chmod(0o600)
    os.environ['HF_TOKEN'] = HF_TOKEN
    # Validate against the HF API so a bad/expired token fails loudly here, not 8 min later.
    import urllib.request, urllib.error, json
    try:
        req = urllib.request.Request('https://huggingface.co/api/whoami-v2',
                                     headers={'Authorization': f'Bearer {HF_TOKEN}'})
        info = json.loads(urllib.request.urlopen(req, timeout=5).read())
        print(f"HF authenticated as: {info.get('name')} ({info.get('email','no-email')})")
    except urllib.error.HTTPError as e:
        print(f"⚠ HF token rejected ({e.code}). Update via Secrets panel.")
else:
    print('No HF_TOKEN secret set. Non-gated models will still work; gated ones will fail with 401.')
    print('To enable: Secrets panel → add HF_TOKEN → enable Notebook access → re-run this cell.')

## Smoke suite (1 model, ~8 min, free)

In [ ]:
!cd /content/tpu && PYTHONPATH=. python benchmarks/harness.py --suite smoke --device tpu --probes default

## Quick suite (5 models, ~50 min, free) — uncomment to run

In [ ]:
# !cd /content/tpu && PYTHONPATH=. python benchmarks/harness.py --suite quick --device tpu --probes default

## Inspect results

In [ ]:
import json
from pathlib import Path

# Latest 5 results
runs_path = Path('/content/tpu/results/runs.jsonl')
if runs_path.exists():
    lines = runs_path.read_text().strip().splitlines()
    for line in lines[-5:]:
        r = json.loads(line)
        print(f"{r.get('model','?')} | {r.get('device','?')} | p50={r.get('latency_p50_ms','?')}ms | tp={r.get('throughput_mean_samples_sec','?')} smp/s")
else:
    print('No runs.jsonl yet — run the smoke cell above first.')

# Per-run probe outputs
run_logs = Path('/content/tpu/results/run_logs')
if run_logs.exists():
    for d in sorted(run_logs.iterdir())[-3:]:
        print(f"\n{d.name}:")
        for f in d.iterdir():
            print(f"  {f.name} ({f.stat().st_size} bytes)")

## Download results for local Grafana replay (ADR-016)

In [ ]:
import shutil, zipfile, os
# Build zip of results/otel/ then add runs.jsonl + run_logs/ for full local replay
zip_path = '/content/tpu_bench_results.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    results_root = '/content/tpu/results'
    for root, _, files in os.walk(results_root):
        for fn in files:
            full = os.path.join(root, fn)
            arc = os.path.relpath(full, results_root)
            z.write(full, arc)

print(f'Wrote {zip_path} ({os.path.getsize(zip_path)} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Auto-download unavailable:', e)
    print('Use the file browser (left sidebar) to download', zip_path)
# On your laptop: unzip into ./results/, then ./scripts/otel_view.sh

## Cost-aware comparison

| Workload          | Colab Pro TPU   | GCP v5e-1 preemptible | Notes                                       |
|-------------------|-----------------|-----------------------|---------------------------------------------|
| Smoke (~8 min)    | $0 marginal     | ~$0.05                | Dev iteration — use Colab                   |
| Quick (~50 min)   | $0 marginal     | ~$0.30                | Dev iteration — use Colab                   |
| Measurement-grade | Not reproducible| controlled v5e-1      | Use v5e-1: known hardware, no co-tenancy    |

Colab Pro subscription is a sunk cost; runs here have $0 marginal cost. Spend on v5e-1
only when you need measurement-grade reproducibility (controlled hardware, controlled env).